# Leiden clustering for TNBC1
This notebook loads per-patient cell features from HDF5, runs PCA, builds a kNN graph,
clusters with Leiden, summarizes clusters, chooses exemplars, and saves results.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from clearit.config import EMBEDDINGS_DIR, OUTPUTS_DIR

from clearit.leiden.io import inspect_hdf5, list_patients, load_hdf5_split
from clearit.leiden.preprocess import standardize_and_pca
from clearit.leiden.graph import build_knn_graph
from clearit.leiden.cluster import leiden
from clearit.leiden.exemplars import select_exemplars

# Reproducibility
SEED = 13

# Paths for both datasets
H5_TNBC1 = EMBEDDINGS_DIR / "TNBC1-MxIF8"  / "inForm_MC7"     / "01_features-expressions" / "tnbc1-mxif8.hdf5"

OUT_DIR = OUTPUTS_DIR / "leiden"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Clustering parameters
PCA_DIMS   = 64
KNN_K      = 60
LEI_RES    = 0.7

In [ ]:
DATASET = "TNBC1"
h5_path = H5_TNBC1
assert h5_path.exists(), f"Missing file: {h5_path}"

# Inspect structure (optional)
summary = inspect_hdf5(h5_path)
print(f"Found {len(summary['groups'])} groups and {len(summary['datasets'])} datasets.")

Found 62 groups and 186 datasets.


In [54]:
# Discover patients and define a train/test split.
available = list_patients(h5_path)
print(f"{DATASET}: {len(available)} patient groups.")

# Here we use first 47 as train and the rest as test for TNBC1-like counts.
TRAIN_PTS = [p for p in available if int(p[1:]) <= 47]
TEST_PTS  = [p for p in available if p not in TRAIN_PTS]

SPLIT = "train"  # "train" or "test"
patients_to_load = TRAIN_PTS if SPLIT == "train" else TEST_PTS
print(f"Using {SPLIT} split with {len(patients_to_load)} patients.")

TNBC1: 62 patient groups.
Using train split with 47 patients.


In [55]:
# Load data with optional caps for large datasets.
X, meta, E, y = load_hdf5_split(
    h5_path=h5_path,
    patients=patients_to_load,
    load_expressions=False,
    load_labels=False,
    per_patient_cap=10_000,
    global_cap=150_000,
    seed=SEED,
)
print(f"Loaded features: X shape = {X.shape}, meta rows = {len(meta)}")
display(meta.head())

Loaded features: X shape = (150000, 256), meta rows = 150000


,patient,idx_within_patient
0,P01,2943
1,P01,3040
2,P01,18783
3,P01,9582
4,P01,24539


In [56]:
# Standardize and PCA
X_pca, scaler, pca = standardize_and_pca(X, n_components=PCA_DIMS, seed=SEED)
print(f"PCA: first 10 EVR = {np.round(pca.explained_variance_ratio_[:10], 4)}")
print(f"PCA: cumulative (k={PCA_DIMS}) = {pca.explained_variance_ratio_[:PCA_DIMS].sum():.4f}")

PCA: first 10 EVR = [0.1496 0.1034 0.0696 0.0687 0.0537 0.0438 0.0373 0.0313 0.0284 0.0242]
PCA: cumulative (k=64) = 0.9575


In [57]:
# Build kNN graph and run Leiden
g = build_knn_graph(X_pca, k=KNN_K, metric="euclidean")
labels, n_clusters = leiden(g, resolution=LEI_RES, seed=SEED)

meta = meta.copy()
meta["cluster_id"] = labels
print(f"Leiden: {n_clusters} clusters")

Leiden: 16 clusters


In [58]:
# Quick cluster summary
cluster_sizes = meta["cluster_id"].value_counts().sort_index()
summary = (
    meta.groupby("cluster_id")["patient"]
    .nunique()
    .rename("n_patients")
    .to_frame()
    .assign(size=cluster_sizes.values)
    .reset_index()
    .sort_values("size", ascending=False)
)
print(f"Total clusters: {summary.shape[0]}")
display(summary)

Total clusters: 16


,cluster_id,n_patients,size
0,0,16,17991
1,1,16,16828
2,2,15,14620
3,3,15,14352
4,4,16,13216
5,5,15,12194
6,6,12,10870
7,7,14,9896
8,8,14,7496
9,9,13,7062


In [59]:
# Exemplar selection
EXEMPLARS_PER_CLUSTER = 10
EXEMPLARS_PER_PATIENT_MAX = 2
DENSITY_K = 15

exemplars = select_exemplars(
    X_pca=X_pca,
    meta=meta,
    labels=labels,
    exemplars_per_cluster=EXEMPLARS_PER_CLUSTER,
    per_patient_max=EXEMPLARS_PER_PATIENT_MAX,
    density_k=DENSITY_K,
)
print(
    f"Selected {len(exemplars)} exemplars across "
    f"{exemplars['cluster_id'].nunique()} clusters."
)
display(exemplars.head(20))

Selected 160 exemplars across 16 clusters.


,patient,idx_within_patient,cluster_id,exemplar_rank
45544,P05,30505,0,1.0
43328,P05,24246,0,2.0
54755,P06,26065,0,14.0
57849,P06,3614,0,23.0
122692,P13,5386,0,45.0
103339,P11,3339,0,47.0
79429,P08,2744,0,56.0
146808,P16,6767,0,60.0
103352,P11,3352,0,62.0
73657,P08,4366,0,72.0


In [60]:
from sklearn.metrics import adjusted_rand_score
import numpy as np

def leiden_stability(X_pca, k, res, n_runs=5):
    g = build_knn_graph(X_pca, k=k)
    labels_list = [leiden(g, resolution=res, seed=i)[0] for i in range(n_runs)]
    pairs = [(i, j) for i in range(n_runs) for j in range(i+1, n_runs)]
    aris = [adjusted_rand_score(labels_list[i], labels_list[j]) for i,j in pairs]
    return np.mean(aris)


In [62]:
leiden_stability(X_pca,KNN_K,LEI_RES)

0.8661321991574825

In [ ]:
# Save outputs
assign_csv    = OUT_DIR / f"{DATASET.lower()}_{SPLIT}_clusters.csv"
exemplars_csv = OUT_DIR / f"{DATASET.lower()}_{SPLIT}_exemplars.csv"

meta.to_csv(assign_csv, index=False)
exemplars.to_csv(exemplars_csv, index=False)

print("Saved:")
print(" -", assign_csv)
print(" -", exemplars_csv)


Saved:
 - /workspace/files/CLEAR-IT/outputs/leiden/tnbc1_train_clusters.csv
 - /workspace/files/CLEAR-IT/outputs/leiden/tnbc1_train_exemplars.csv


In [ ]:
#!/usr/bin/env python3
"""
Scan kNN-k and Leiden resolution; report cluster count, stability (mean ARI), and quality.

Outputs:
  - CSV with all runs: <OUTPUTS_DIR>/leiden/<dataset>_<split>_scan.csv
  - Console summary of settings closest to target cluster count (default: 14)

Dependencies:
  clearit.leiden.{io, preprocess, graph}
  numpy, pandas, scikit-learn, igraph, leidenalg
"""

from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple
import itertools
import numpy as np
import pandas as pd

import igraph as ig
import leidenalg as la
from sklearn.metrics import adjusted_rand_score

from clearit.config import EMBEDDINGS_DIR, OUTPUTS_DIR
from clearit.leiden.io import list_patients, load_hdf5_split
from clearit.leiden.preprocess import standardize_and_pca
from clearit.leiden.graph import build_knn_graph


# ----------------------------- Configuration -----------------------------

# Dataset selection
DATASET = "TNBC1"
SPLIT = "train"    # "train" or "test"

# HDF5 paths
H5_TNBC1 = EMBEDDINGS_DIR / "TNBC1-MxIF8"  / "inForm_MC7"    / "01_features-expressions" / "tnbc1-mxif8.hdf5"

# Embedding + graph defaults (centered on your current working point)
PCA_DIMS_DEFAULT = 64
K_GRID = [40, 50, 60, 70, 80]                  # includes 60
RES_GRID = [0.5, 0.6, 0.7, 0.8, 0.9]           # includes 0.7
SEEDS = [0, 1, 2, 3, 4]                        # used for stability
TARGET_CLUSTERS = 14

# Subsampling caps (set to None for full data)
PER_PATIENT_CAP = 10_000
GLOBAL_CAP = 150_000

# Reproducibility
SEED = 42

# Output
OUT_DIR = OUTPUTS_DIR / "leiden"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------- Utilities -------------------------------

@dataclass
class PartitionStats:
    labels: np.ndarray
    quality: float
    n_clusters: int


def run_leiden_partition(
    g: ig.Graph,
    resolution: float,
    seed: int,
) -> PartitionStats:
    """
    Run Leiden with RBConfigurationVertexPartition; return labels, quality, and cluster count.
    """
    part = la.find_partition(
        g,
        la.RBConfigurationVertexPartition,
        weights=g.es["weight"] if "weight" in g.es.attributes() else None,
        resolution_parameter=resolution,
        seed=seed,
    )
    labels = np.asarray(part.membership, dtype=int)
    quality = float(part.quality())  # objective value for the chosen partition type
    n_clusters = int(labels.max() + 1) if labels.size else 0
    return PartitionStats(labels=labels, quality=quality, n_clusters=n_clusters)


def mean_pairwise_ari(labels_list: List[np.ndarray]) -> float:
    """
    Compute mean Adjusted Rand Index over all unique pairs in a list of labelings.
    """
    if len(labels_list) < 2:
        return 1.0
    pairs = [(i, j) for i in range(len(labels_list)) for j in range(i + 1, len(labels_list))]
    aris = [adjusted_rand_score(labels_list[i], labels_list[j]) for i, j in pairs]
    return float(np.mean(aris))


def choose_split_patients(h5_path: Path) -> Tuple[list[str], list[str]]:
    """
    Build a simple train/test split over available patient groups.
    For TNBC1-like counts, the first 47 patients are train; remainder are test.
    For other counts, split 75/25 by index.
    """
    pts = list_patients(h5_path)
    if len(pts) >= 63:
        train = [p for p in pts if int(p[1:]) <= 47]
        test = [p for p in pts if p not in train]
        return train, test
    # Generic fallback split
    n_train = max(1, int(0.75 * len(pts)))
    return pts[:n_train], pts[n_train:]


# --------------------------------- Main ----------------------------------

def main():
    # Resolve dataset path
    h5_path = H5_TNBC1
    assert h5_path.exists(), f"Missing file: {h5_path}"

    # Determine patients for chosen split
    train_pts, test_pts = choose_split_patients(h5_path)
    patients = train_pts if SPLIT == "train" else test_pts
    print(f"{DATASET} | {SPLIT}: {len(patients)} patients")

    # Load and embed once; graph is rebuilt per k
    X, meta, _, _ = load_hdf5_split(
        h5_path=h5_path,
        patients=patients,
        load_expressions=False,
        load_labels=False,
        per_patient_cap=PER_PATIENT_CAP,
        global_cap=GLOBAL_CAP,
        seed=SEED,
    )
    print(f"Features loaded: X = {X.shape}, meta rows = {len(meta)}")

    X_pca, scaler, pca = standardize_and_pca(X, n_components=PCA_DIMS_DEFAULT, seed=SEED)
    cum_var = float(pca.explained_variance_ratio_[:PCA_DIMS_DEFAULT].sum())
    print(f"PCA dims = {PCA_DIMS_DEFAULT}, cumulative explained variance = {cum_var:.4f}")

    # Scan grid
    rows = []
    for k, res in itertools.product(K_GRID, RES_GRID):
        # Build kNN graph for this k
        g = build_knn_graph(X_pca, k=k, metric="euclidean")

        # Precompute some graph stats for reference
        n = g.vcount()
        m = g.ecount()
        mean_deg = float(np.mean(g.degree())) if n > 0 else 0.0
        n_components = len(g.components())

        # Multiple seeds for stability
        label_runs = []
        qualities = []
        ncls = []
        for s in SEEDS:
            stats = run_leiden_partition(g, resolution=res, seed=s)
            label_runs.append(stats.labels)
            qualities.append(stats.quality)
            ncls.append(stats.n_clusters)

        # Stability and summary metrics
        mean_ari = mean_pairwise_ari(label_runs)
        mean_clusters = float(np.mean(ncls))
        std_clusters = float(np.std(ncls))
        mean_quality = float(np.mean(qualities))
        std_quality = float(np.std(qualities))

        rows.append(
            dict(
                dataset=DATASET,
                split=SPLIT,
                pca_dims=PCA_DIMS_DEFAULT,
                k=k,
                resolution=res,
                seeds=len(SEEDS),
                mean_clusters=mean_clusters,
                std_clusters=std_clusters,
                mean_ari=mean_ari,
                mean_quality=mean_quality,
                std_quality=std_quality,
                n_vertices=n,
                n_edges=m,
                mean_degree=mean_deg,
                n_components=n_components,
                target=TARGET_CLUSTERS,
                abs_diff_from_target=abs(mean_clusters - TARGET_CLUSTERS),
            )
        )
        print(
            f"k={k:>2}, res={res:>3.1f} -> "
            f"clusters {mean_clusters:.1f}±{std_clusters:.1f}, "
            f"ARI={mean_ari:.3f}, quality={mean_quality:.4f}, "
            f"|Δ|={abs(mean_clusters - TARGET_CLUSTERS):.1f}"
        )

    df = pd.DataFrame(rows).sort_values(
        ["abs_diff_from_target", "mean_ari", "mean_quality"],
        ascending=[True, False, False],
    ).reset_index(drop=True)

    # Save results
    out_csv = OUT_DIR / f"{DATASET.lower()}_{SPLIT}_scan.csv"
    df.to_csv(out_csv, index=False)
    print(f"\nSaved scan results: {out_csv}")

    # Console summary: top candidates near target cluster count
    top = df.head(10)[
        [
            "k",
            "resolution",
            "mean_clusters",
            "std_clusters",
            "mean_ari",
            "mean_quality",
            "mean_degree",
            "n_components",
        ]
    ]
    print("\nTop candidates near target cluster count:")
    print(top.to_string(index=False))


if __name__ == "__main__":
    main()


TNBC1 | train: 46 patients
Features loaded: X = (150000, 256), meta rows = 150000
PCA dims = 64, cumulative explained variance = 0.9575
k=40, res=0.5 -> clusters 14.0±0.0, ARI=0.782, quality=708945.9966, |Δ|=0.0
k=40, res=0.6 -> clusters 14.8±0.7, ARI=0.775, quality=700688.6298, |Δ|=0.8
k=40, res=0.7 -> clusters 16.0±0.6, ARI=0.795, quality=694217.6134, |Δ|=2.0
k=40, res=0.8 -> clusters 17.6±1.2, ARI=0.780, quality=687677.3077, |Δ|=3.6
k=40, res=0.9 -> clusters 19.2±0.4, ARI=0.846, quality=681827.0113, |Δ|=5.2
k=50, res=0.5 -> clusters 13.6±0.5, ARI=0.825, quality=868093.3908, |Δ|=0.4
k=50, res=0.6 -> clusters 14.2±0.4, ARI=0.850, quality=858789.7177, |Δ|=0.2
k=50, res=0.7 -> clusters 14.6±0.8, ARI=0.824, quality=849337.5390, |Δ|=0.6
k=50, res=0.8 -> clusters 15.6±1.0, ARI=0.823, quality=840358.8715, |Δ|=1.6
k=50, res=0.9 -> clusters 17.6±0.8, ARI=0.758, quality=832152.1226, |Δ|=3.6
k=60, res=0.5 -> clusters 13.4±0.5, ARI=0.788, quality=1023501.2671, |Δ|=0.6
k=60, res=0.6 -> clusters 1